In [1]:
import requests
from bs4 import BeautifulSoup

# 1. Get HTML

In [2]:
url = "https://tillschwoerer.github.io/webscraping-demo/"
html = requests.get(url).text
html

'<!DOCTYPE html>\n<html lang="en">\n\n<head>\n  <meta charset="UTF-8">\n  <title>SMA</title>\n  <link rel="stylesheet" href="styles.css">\n</head>\n\n<body>\n\n  <img src="https://upload.wikimedia.org/wikipedia/de/1/13/HAW_Kiel_Logo.svg" alt="Logo of HAW Kiel"\n    title="HAW Kiel logo" width="300" align="right">\n\n  <h1 class="header">Welcome to Tools and Programming Languages for Data Science!</h1>\n\n  <section id="topics">\n\n    <p>Overview of course topics:</p>\n\n    <div class="topic-box" id="topic-analysis">\n      <h2 class="header topic">Data Analysis</h2>\n      <p class="description">We learn how to use the Pandas library for analyse and manipulate data. </p>\n      <ul>\n        <li>Cleaning data</li>\n        <li>Exploring data</li>\n        <li>Data Visualization</li>\n      </ul>\n      <p class="lecture-date">16.03.2026</p>\n    </div>\n\n    <div class="topic-box" id="topic-io">\n      <h2 class="header topic">Data Input and Output</h2>\n      <p class="description"

# 2. Parse HTML

In [3]:
soup = BeautifulSoup(html, "html.parser")

# 3. Locate elements (select)

- The `select_one` method selects the first matching element
- the `select` method selects all matching elements, returning a list.

## 3.1 Simple selectors

### ID (unique)

In [4]:
overview = soup.select_one("#topic-analysis")
overview

<div class="topic-box" id="topic-analysis">
<h2 class="header topic">Data Analysis</h2>
<p class="description">We learn how to use the Pandas library for analyse and manipulate data. </p>
<ul>
<li>Cleaning data</li>
<li>Exploring data</li>
<li>Data Visualization</li>
</ul>
<p class="lecture-date">16.03.2026</p>
</div>

In [5]:
overview = soup.select("#topic-analysis")
overview

[<div class="topic-box" id="topic-analysis">
 <h2 class="header topic">Data Analysis</h2>
 <p class="description">We learn how to use the Pandas library for analyse and manipulate data. </p>
 <ul>
 <li>Cleaning data</li>
 <li>Exploring data</li>
 <li>Data Visualization</li>
 </ul>
 <p class="lecture-date">16.03.2026</p>
 </div>]

### Class (not unique)

In [6]:
soup.select_one('.topic')

<h2 class="header topic">Data Analysis</h2>

In [7]:
soup.select('.topic')

[<h2 class="header topic">Data Analysis</h2>,
 <h2 class="header topic">Data Input and Output</h2>,
 <h2 class="header topic">Version Control with Git and Github</h2>]

### HTML tag

In [9]:
soup.select('h1')

[<h1 class="header">Welcome to Tools and Programming Languages for Data Science!</h1>]

### Attributes

In [10]:
soup.select('[target="_blank"]')

[<a class="external-link" href="https://moduldatenbank.fh-kiel.de/de-DE/Module/Details/bcd9db4d-7dc1-497e-ac51-6959a1defc7d?versionId=0" target="_blank">Module Database</a>]

## 3.2 Hierarchical selectors

### Child (>)
A child element is directly contained within another element

In [19]:
soup.select("section > p") 

[<p>Overview of course topics:</p>]

### Descendant
Any element nested inside another element, at any level (child, grandchild, ...)

In [20]:
soup.select("section p")

[<p>Overview of course topics:</p>,
 <p class="description">We learn how to use the Pandas library for analyse and manipulate data. </p>,
 <p class="lecture-date">01.09.2025</p>,
 <p class="description">We learn how to read and write data from or to different sources </p>,
 <p class="note">Extra note: demo for descendant selectors.</p>,
 <p class="lecture-date">01.10.2025</p>,
 <p class="description">We learn how to work productively with Git and Github.</p>,
 <p class="lecture-date">01.11.2025</p>]

### Siblings
Siblings are elements that share the same parent

In [22]:
soup.select("h2 ~ p")   # any p elements that are siblings of h2

[<p class="description">We learn how to use the Pandas library for analyse and manipulate data. </p>,
 <p class="lecture-date">01.09.2025</p>,
 <p class="description">We learn how to read and write data from or to different sources </p>,
 <p class="lecture-date">01.10.2025</p>,
 <p class="description">We learn how to work productively with Git and Github.</p>,
 <p class="lecture-date">01.11.2025</p>,
 <p>Visit <a class="external-link" href="https://learn.fh-kiel.de/course/view.php?id=16727">Moodle</a> or <a class="external-link" href="https://moduldatenbank.fh-kiel.de/de-DE/Module/Details/bcd9db4d-7dc1-497e-ac51-6959a1defc7d?versionId=0" target="_blank">Module Database</a> for more information.</p>]

### Adjacent sibling

In [ ]:
soup.select("h2 + p") # p elements that immediately follow an h2 element, and share the same parent

### Full CSS Path

In [23]:
soup.select_one('#topic-io > ul > li:nth-child(3)')

<li>Web Scraping</li>

## 3.3 Alternative: find instead of select

- The method `find` is an alternative to `select_one`
- The method `find_all` is an alternative to `select`
- Equally suited for easy cases, less flexible for complex cases

In [ ]:
soup.find(name='h2').text                  # HTML tag
soup.find(id="topics").text                # ID
soup.find(class_='header').text            # Class
soup.find(attrs={"target": "_blank"}).text # Attributes

One advantage of find: we can search for **strings** or **substrings** (via regex)

In [ ]:
soup.find(string="Web Scraping").text

In [ ]:
import re
soup.find(string=re.compile("Scraping"))

# 4. Access content

We can access different content: often the displayed **text**, but sometimes we are interested in an **HTML attribute**

In [12]:
link = soup.select_one('a')
print(link.get_text())
print(link.text)

Module Database
Module Database


In [ ]:
print(link.get('href'))


https://moduldatenbank.fh-kiel.de/de-DE/Module/Details/bcd9db4d-7dc1-497e-ac51-6959a1defc7d?versionId=0
_blank


# 5. Scraping loop


In [15]:
import time

In [16]:
topics = []
boxes = soup.select(".topic-box")
for box in boxes:
    topic = {}
    topic['topic'] = box.select_one("h2").text
    topic['date'] = box.select_one(".lecture-date").text
    topic['description'] = box.select_one(".description").text
    topic['aspects'] = [item.text for item in box.select("li")]
    topics.append(topic)
    time.sleep(2)

topics

[{'topic': 'Data Analysis',
  'date': '16.03.2026',
  'description': 'We learn how to use the Pandas library for analyse and manipulate data. ',
  'aspects': ['Cleaning data', 'Exploring data', 'Data Visualization']},
 {'topic': 'Data Input and Output',
  'date': '23.03.2026',
  'description': 'We learn how to read and write data from or to different sources ',
  'aspects': ['APIs', 'SQL Databases', 'Web Scraping']},
 {'topic': 'Version Control with Git and Github',
  'date': '30.03.2026',
  'description': 'We learn how to work productively with Git and Github.',
  'aspects': ['Commit workflow',
   'Undo changes',
   'Branching and merging',
   'Collaborating via Github']}]

In [17]:
import pandas as pd
df  = pd.DataFrame(topics)
df

,topic,date,description,aspects
0,Data Analysis,16.03.2026,We learn how to use the Pandas library for ana...,"[Cleaning data, Exploring data, Data Visualiza..."
1,Data Input and Output,23.03.2026,We learn how to read and write data from or to...,"[APIs, SQL Databases, Web Scraping]"
2,Version Control with Git and Github,30.03.2026,We learn how to work productively with Git and...,"[Commit workflow, Undo changes, Branching and ..."


In [50]:
list_of_dfs = pd.read_html("https://tillschwoerer.github.io/webscraping-demo/")
list_of_dfs[0]

,Component,Weight
0,Assignment 1,20%
1,Assignment 2,20%
2,Test,60%
